# Eval & ablations — reproduction du Table 1, Fig. 5/6/7

Notebook d'évaluation à exécuter une fois les 3 runs (`fm-ot`, `fm-vp`, `fm-ddpm`) terminés. Charge les checkpoints finaux, calcule la table principale et produit toutes les figures du rapport.

Hardware : T4 suffisant pour l'éval (sampling + FID), ~60 min total.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath("../src"))

import torch, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

from flow_matching_b3.paths import get_path
from flow_matching_b3.unet import adm_unet_cifar10
from flow_matching_b3.ema import EMA
from flow_matching_b3.sampling import sample, make_vf, euler_sample, midpoint_sample, rk4_sample, dopri5_sample
from flow_matching_b3.fid import compute_fid_cifar10
from flow_matching_b3.train import denorm, find_latest_ckpt, make_grid_png, enable_cuda_perf

device = "cuda" if torch.cuda.is_available() else "cpu"
enable_cuda_perf()  # TF32 + cudnn.benchmark pour un sampling/FID plus rapide

RUNS_ROOT = Path("/content/drive/MyDrive/flow-matching-b3") if "google.colab" in sys.modules else Path("./runs")
RUN_NAMES = {"ot": "fm-ot-cifar10", "vp": "fm-vp-cifar10", "ddpm": "fm-ddpm-cifar10"}
print("runs in:", RUNS_ROOT)

## 1. Charger les 3 modèles (poids EMA)

In [ ]:
def load_ema_model(run_dir: Path):
    model = adm_unet_cifar10().to(device)
    ema = EMA(model)
    state = torch.load(find_latest_ckpt(run_dir), map_location=device)
    model.load_state_dict(state["model"])
    ema.load_state_dict(state["ema"])
    # Replace raw weights with EMA shadow; we only sample from EMA.
    for p, p_ema in zip(model.parameters(), ema.shadow.parameters(), strict=True):
        p.data.copy_(p_ema.data)
    model.eval()
    return model, state["step"]

models = {}
paths = {
    "ot":   get_path("ot",   sigma_min=1e-4),
    "vp":   get_path("vp"),
    "ddpm": get_path("ddpm"),
}
for key, name in RUN_NAMES.items():
    run_dir = RUNS_ROOT / name
    if not (run_dir.exists() and find_latest_ckpt(run_dir)):
        print(f"[skip] no checkpoint for {name}"); continue
    m, step = load_ema_model(run_dir)
    models[key] = m
    print(f"loaded {name} @ step {step}")

## 2. Table 1 — FID(50k) + NFE moyen (dopri5)

In [ ]:
from tqdm.auto import tqdm

@torch.no_grad()
def eval_fid_and_nfe(model, path, n_samples=50_000, batch=128):
    nfes = []
    def gen():
        remaining = n_samples
        with tqdm(total=n_samples) as pbar:
            while remaining > 0:
                b = min(batch, remaining)
                imgs, nfe = sample(model, path, shape=(b, 3, 32, 32), solver="dopri5", device=device)
                nfes.append(nfe)
                pbar.update(b)
                yield denorm(imgs)
                remaining -= b
    fid = compute_fid_cifar10(gen(), n_samples=n_samples)
    return fid, float(np.mean(nfes))

table1 = {}
for key, m in models.items():
    fid, nfe = eval_fid_and_nfe(m, paths[key])
    table1[key] = {"fid": fid, "nfe": nfe}
    print(f"{key:5s}  FID={fid:.3f}  NFE={nfe:.1f}")

with open(RUNS_ROOT / "table1.json", "w") as f:
    json.dump(table1, f, indent=2)

## 3. Fig. 6 — trajectoires de sampling avec mêmes seeds

On échantillonne **les mêmes 8 bruits initiaux** avec les 3 modèles, en visualisant l'état intermédiaire à `t ∈ {0, 1/3, 2/3, 1}`.

In [ ]:
@torch.no_grad()
def euler_trajectory(model, path, x0, t_stops=(0.0, 1/3, 2/3, 1.0), nfe=200):
    vf = make_vf(model, path)
    dt = 1.0 / nfe
    x = x0.clone()
    out = {0.0: x.clone()}
    next_idx = 1
    for k in range(nfe):
        t_val = k * dt
        t = torch.full((x.size(0),), t_val, device=x.device)
        x = x + dt * vf(x, t)
        if next_idx < len(t_stops) and (k + 1) * dt >= t_stops[next_idx] - 1e-6:
            out[t_stops[next_idx]] = x.clone()
            next_idx += 1
    return out

torch.manual_seed(7); x0 = torch.randn(8, 3, 32, 32, device=device)
fig, axes = plt.subplots(len(models), 4, figsize=(12, 3 * len(models)))
if len(models) == 1: axes = axes.reshape(1, -1)
stops = (0.0, 1/3, 2/3, 1.0)
for row, (key, m) in enumerate(models.items()):
    traj = euler_trajectory(m, paths[key], x0, t_stops=stops, nfe=200)
    for col, t in enumerate(stops):
        grid = make_grid_png(traj[t], n_row=8).permute(1, 2, 0).cpu()
        axes[row, col].imshow(grid); axes[row, col].axis("off")
        if row == 0: axes[row, col].set_title(f"t = {t:.2f}")
    axes[row, 0].set_ylabel(key, rotation=0, labelpad=20, fontsize=12)
plt.suptitle("Sampling trajectories from identical seeds (≈ Fig. 6)");
plt.savefig(RUNS_ROOT / "fig6_trajectories.png", dpi=120, bbox_inches="tight"); plt.show()

## 4. Fig. 7 — FID(NFE) avec Euler / Midpoint / RK4

Sur le modèle **FM-OT** uniquement (le résultat phare du papier). FID sur **10 000** samples pour rester rapide — ⚠️ biaisé ↑ vs l'échelle 50k de la Table 1 : à lire en **relatif** (comparaison entre solveurs / NFE), pas en absolu.

In [ ]:
if "ot" not in models:
    print("FM-OT manquant — skip")
else:
    NFE_GRID = [4, 8, 16, 32, 64]
    SOLVERS = {"euler": euler_sample, "midpoint": midpoint_sample, "rk4": rk4_sample}
    fid_curves = {s: [] for s in SOLVERS}
    vf = make_vf(models["ot"], paths["ot"])

    for solver_name, solver_fn in SOLVERS.items():
        for nfe in NFE_GRID:
            def gen():
                remaining = 10_000
                while remaining > 0:
                    b = min(256, remaining); x0 = torch.randn(b, 3, 32, 32, device=device)
                    yield denorm(solver_fn(vf, x0, nfe=nfe))
                    remaining -= b
            f = compute_fid_cifar10(gen(), n_samples=10_000)
            fid_curves[solver_name].append(f)
            print(f"  {solver_name} NFE={nfe} → FID={f:.2f}")

    plt.figure(figsize=(6, 4))
    for s, ys in fid_curves.items():
        plt.plot(NFE_GRID, ys, marker="o", label=s)
    plt.xlabel("NFE"); plt.ylabel("FID"); plt.title("FM-OT — FID vs NFE (≈ Fig. 7)")
    plt.legend(); plt.grid(alpha=0.3); plt.xscale("log")
    plt.savefig(RUNS_ROOT / "fig7_fid_nfe.png", dpi=120, bbox_inches="tight"); plt.show()
    json.dump({"nfe_grid": NFE_GRID, **fid_curves}, open(RUNS_ROOT / "fig7_fid_nfe.json", "w"), indent=2)

## 5. Export Markdown des résultats (à coller dans le rapport)

In [ ]:
ref = {"ot": 6.35, "vp": 8.06, "ddpm": 7.48}  # paper Table 1 (CIFAR-10) — FID legacy TF-Inception
lines = ["| Modèle | FID (notre) | FID (papier) | NFE moyen (dopri5) |", "|---|---|---|---|"]
for key in ["ddpm", "vp", "ot"]:
    if key in table1:
        r = table1[key]
        lines.append(f"| {key.upper()} | {r['fid']:.2f} | {ref[key]} | {r['nfe']:.1f} |")
print("\n".join(lines))
print("\n_FID en clean-fid mode=legacy_tensorflow sur 50k samples — protocole comparable au papier._")